<a href="https://colab.research.google.com/github/alejitovm97-byte/Alejo-Varelas-projects/blob/gh-pages/07_fases3_cruces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/'
            'TFM_regime_allocation/v2')
DIRS = {k: ROOT / v for k, v in {
    'raw': 'data/raw',
    'processed': 'data/processed',
    'results': 'results',
    'logs': 'logs'}.items()}

P = DIRS['processed']
pit = pd.read_parquet(P / 'f2_factores.parquet')
ret = pd.read_parquet(P / 'f2_retornos_m.parquet')
cx  = pd.read_parquet(P / 'b6_clock_expost.parquet')

FACT  = ['value','quality','momentum','growth','lowvol']
ORDEN = ['Reflation','Recovery','Overheat','Stagflation']
MIN_N = 50

cl = cx[['fase']].copy()
cl.index.name = 'mes'
cl = cl.reset_index()

d = pit.merge(cl, on='mes', how='inner')
m = d[['mes','fase']].drop_duplicates()
m = m.sort_values('mes').reset_index(drop=True)
m['ep'] = (m.fase != m.fase.shift()).cumsum()
d = d.merge(m[['mes','ep']], on='mes', how='left')

print(f'filas {len(d):,} | meses {d.mes.nunique()}')
print(f'rango {d.mes.min().date()} -> {d.mes.max().date()}')

dur = m.groupby('ep').size()
r = m.groupby('fase').agg(meses=('mes','size'),
                          episodios=('ep','nunique'))
r['dur'] = m.groupby('fase')['ep'].apply(
    lambda s: dur[s.unique()].mean())
print()
print(r.reindex(ORDEN).round(1).to_string())
print(f'\nTOTAL episodios: {m.ep.nunique()}')

Mounted at /content/drive
filas 169,531 | meses 346
rango 1997-01-31 -> 2025-10-31

             meses  episodios   dur
fase                               
Reflation       72          7  10.3
Recovery        94          8  11.8
Overheat       127          7  18.1
Stagflation     53          7   7.6

TOTAL episodios: 29


In [ ]:
#Bloque 1 — carga y verificación:

from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path('/content/drive/MyDrive/'
            'TFM_regime_allocation/v2')
DIRS = {k: ROOT / v for k, v in {
    'raw': 'data/raw',
    'processed': 'data/processed',
    'results': 'results',
    'logs': 'logs'}.items()}
P = DIRS['processed']

pit = pd.read_parquet(P / 'f2_factores.parquet')
ret = pd.read_parquet(P / 'f2_retornos_m.parquet')
cx  = pd.read_parquet(P / 'b6_clock_expost.parquet')

FACT  = ['value','quality','momentum',
         'growth','lowvol']
ORDEN = ['Reflation','Recovery',
         'Overheat','Stagflation']
MIN_N = 50

print('pit  ', pit.shape)
print('ret  ', ret.shape)
print('clock', cx.shape)
print('\ncobertura de factores:')
print(pit[FACT].notna().mean().round(3).to_string())
print('\nreloj ex-post:', cx.index.min().date(),
      '->', cx.index.max().date())
print(cx.fase.value_counts().to_string())

pit   (174058, 111)
ret   (290188, 5)
clock (916, 7)

cobertura de factores:
value       0.967
quality     0.978
momentum    0.988
growth      0.857
lowvol      0.989

reloj ex-post: 1949-07-31 -> 2025-10-31
fase
Overheat       305
Recovery       239
Reflation      204
Stagflation    168


In [ ]:
#Bloque 2 — panel cruzado con fase, episodio y retorno forward. Esta vez d se arma siempre desde pit, nunca mutando el resultado anterior, así que podés correrlo las veces que quieras sin que se acumulen columnas:

cl = cx[['fase']].copy()
cl.index.name = 'mes'
cl = cl.reset_index()

r2 = ret.sort_values(['ric','mes']).copy()
r2['fwd1'] = r2.groupby('ric')['ret_total'].shift(-1)

d = pit.merge(cl, on='mes', how='inner')
d = d.merge(r2[['ric','mes','fwd1']],
            on=['ric','mes'], how='left')

m = d[['mes','fase']].drop_duplicates()
m = m.sort_values('mes').reset_index(drop=True)
m['ep'] = (m.fase != m.fase.shift()).cumsum()
d = d.merge(m[['mes','ep']], on='mes', how='left')

print(f'filas {len(d):,} | meses {d.mes.nunique()}')
print(f'rango {d.mes.min().date()} -> '
      f'{d.mes.max().date()}')
print(f'con fwd1 {int(d.fwd1.notna().sum()):,}')
print(f'duplicados {d.duplicated(["ric","mes"]).sum()}')

dur = m.groupby('ep').size()
r = m.groupby('fase').agg(
    meses=('mes','size'),
    episodios=('ep','nunique'))
r['dur'] = m.groupby('fase')['ep'].apply(
    lambda s: dur[s.unique()].mean())
print()
print(r.reindex(ORDEN).round(1).to_string())
print(f'\nTOTAL episodios: {m.ep.nunique()}')

filas 169,531 | meses 346
rango 1997-01-31 -> 2025-10-31
con fwd1 168,382
duplicados 0

             meses  episodios   dur
fase                               
Reflation       72          7  10.3
Recovery        94          8  11.8
Overheat       127          7  18.1
Stagflation     53          7   7.6

TOTAL episodios: 29


In [ ]:
#Bloque 3 — carteras por quintil en las dos ponderaciones, con la verificación incondicional:

def carteras(df, f, y='fwd1', peso=None):
    s = df.dropna(subset=[f, y]).copy()
    n = s.groupby('mes')[f].transform('size')
    s = s[n >= MIN_N]
    s['q'] = s.groupby('mes')[f].transform(
        lambda x: pd.qcut(x, 5, labels=False,
                          duplicates='drop') + 1)
    if peso is None:
        return (s.groupby(['mes','q'])[y]
                 .mean().unstack())
    s['w'] = s[peso].where(s[peso] > 0)
    s = s.dropna(subset=['w'])
    s['x'] = s[y] * s.w
    return (s.groupby(['mes','q'])
             .apply(lambda z: z.x.sum()/z.w.sum())
             .unstack())

SP = {'equi': {}, 'cap': {}}
for etq, peso in [('equi', None), ('cap', 'mc_live')]:
    print(f'\n=== {etq.upper()} ===')
    for f in FACT:
        rq = carteras(d, f, peso=peso)
        sp = (rq[5] - rq[1]).dropna()
        SP[etq][f] = sp
        mu = rq.mean() * 12
        t = sp.mean() / (sp.std() / np.sqrt(len(sp)))
        print(f'{f:9s}',
              ' '.join(f'{v:6.2f}' for v in mu.values),
              f'| Q5-Q1 {sp.mean()*12:6.2f} t={t:5.2f}')


=== EQUI ===
value      10.43  10.39  12.04  11.98  12.96 | Q5-Q1   2.53 t= 0.92
quality    10.98  10.79  11.56  11.72  12.95 | Q5-Q1   1.97 t= 0.71
momentum   10.53  11.97  11.31  12.44  11.87 | Q5-Q1   1.34 t= 0.32
growth     11.59  11.81  11.18  11.59  11.79 | Q5-Q1   0.19 t= 0.09
lowvol     12.08  12.33  12.00  11.48  10.20 | Q5-Q1  -1.88 t=-0.45

=== CAP ===


/tmp/ipykernel_501/536424770.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda z: z.x.sum()/z.w.sum())


value       7.99  10.55  10.46  10.21  12.34 | Q5-Q1   4.35 t= 1.45


/tmp/ipykernel_501/536424770.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda z: z.x.sum()/z.w.sum())


quality     2.57   6.87   8.65  10.64  13.15 | Q5-Q1  10.58 t= 3.20


/tmp/ipykernel_501/536424770.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda z: z.x.sum()/z.w.sum())


momentum    4.69  11.15   9.88   9.43  10.96 | Q5-Q1   6.27 t= 1.45


/tmp/ipykernel_501/536424770.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda z: z.x.sum()/z.w.sum())


growth      7.78   7.36   9.63  11.14  12.40 | Q5-Q1   4.62 t= 1.77
lowvol      9.53   9.31  10.13  10.84   9.26 | Q5-Q1  -0.26 t=-0.06


/tmp/ipykernel_501/536424770.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda z: z.x.sum()/z.w.sum())


In [ ]:
#Bloque 4 — la tabla condicionada, en las dos ponderaciones:

mm = d[['mes','fase','ep']].drop_duplicates()
mm = mm.set_index('mes')

def tabla(spd):
    filas = []
    for f in FACT:
        x = pd.DataFrame({'sp': spd[f]}).join(mm)
        x = x.dropna(subset=['fase'])
        for fase in ORDEN:
            g = x[x.fase == fase]
            pe = g.groupby('ep').sp.mean() * 12
            filas.append({'factor': f, 'fase': fase,
                'mes_m': g.sp.mean() * 12,
                'ep_m': pe.mean(),
                'ep_sd': pe.std(),
                'ep_n': g.ep.nunique(),
                'ep_pos': int((pe > 0).sum())})
    return pd.DataFrame(filas)

def W(res, c):
    return (res.pivot(index='fase', columns='factor',
            values=c).reindex(ORDEN)[FACT].round(2))

T = {k: tabla(SP[k]) for k in ['equi', 'cap']}

for k in ['equi', 'cap']:
    print('\n' + '=' * 66)
    print(f'{k.upper()} — media entre episodios (%/anio)')
    print('=' * 66)
    print(W(T[k], 'ep_m').to_string())
    print('\nmedia sobre meses (contraste):')
    print(W(T[k], 'mes_m').to_string())
    print('\nepisodios positivos / total:')
    p = T[k].copy()
    p['x'] = (p.ep_pos.astype(str) + '/' +
              p.ep_n.astype(str))
    print(p.pivot(index='fase', columns='factor',
          values='x').reindex(ORDEN)[FACT].to_string())

print('\n' + '=' * 66)
print('ROBUSTEZ: coinciden en signo las dos ponderaciones')
print('=' * 66)
sig = (np.sign(W(T['equi'], 'ep_m')) ==
       np.sign(W(T['cap'], 'ep_m')))
print(sig.to_string())


EQUI — media entre episodios (%/anio)
factor       value  quality  momentum  growth  lowvol
fase                                                 
Reflation     1.87     6.87      1.48   -2.81   12.16
Recovery      1.28    -8.47     -2.47   -4.92  -11.51
Overheat      2.86     0.48      0.13    1.02   -7.68
Stagflation  -0.44     3.40      8.18    3.77   -0.95

media sobre meses (contraste):
factor       value  quality  momentum  growth  lowvol
fase                                                 
Reflation    11.98     3.94     -6.25   -7.45   11.46
Recovery     -0.97    -1.95      4.37   -0.03   -4.49
Overheat      2.46     0.27     -0.10    0.08   -9.20
Stagflation  -3.95    10.33      9.72   11.25    2.13

episodios positivos / total:
factor      value quality momentum growth lowvol
fase                                            
Reflation     4/7     4/7      4/7    4/7    7/7
Recovery      4/8     4/8      6/8    2/8    4/8
Overheat      5/7     4/7      4/7    4/7    1/7
Stagfl

In [ ]:
#Bloque 5 — análisis por eje, con las predicciones ya declaradas:

SUBE_G = ['Recovery', 'Overheat']
SUBE_I = ['Overheat', 'Stagflation']

mm2 = d[['mes','fase']].drop_duplicates()
mm2 = mm2.sort_values('mes').reset_index(drop=True)
mm2['crec'] = np.where(
    mm2.fase.isin(SUBE_G), 'sube', 'baja')
mm2['infl'] = np.where(
    mm2.fase.isin(SUBE_I), 'sube', 'baja')
for e in ['crec', 'infl']:
    mm2['ep_' + e] = (
        mm2[e] != mm2[e].shift()).cumsum()
mm2 = mm2.set_index('mes')

for e in ['crec', 'infl']:
    dur = mm2.groupby('ep_' + e).size()
    print(f"eje {e}: {mm2['ep_'+e].nunique()} episodios "
          f"| duracion media {dur.mean():.1f} meses")
    print(mm2[e].value_counts().to_string(), '\n')

def tabla_eje(spd, eje):
    filas = []
    for f in FACT:
        x = pd.DataFrame({'sp': spd[f]}).join(mm2)
        x = x.dropna(subset=[eje])
        for est in ['sube', 'baja']:
            g = x[x[eje] == est]
            pe = g.groupby('ep_' + eje).sp.mean() * 12
            filas.append({'factor': f, 'estado': est,
                'ep_m': pe.mean(), 'ep_n': len(pe),
                'ep_pos': int((pe > 0).sum()),
                'mes_m': g.sp.mean() * 12})
    return pd.DataFrame(filas)

for eje, nom in [('crec','CRECIMIENTO'),
                 ('infl','INFLACION')]:
    print('=' * 62)
    print(f'EJE {nom}')
    print('=' * 62)
    for k in ['equi', 'cap']:
        t = tabla_eje(SP[k], eje)
        w = t.pivot(index='estado', columns='factor',
              values='ep_m').reindex(['sube','baja'])[FACT]
        p = t.copy()
        p['x'] = (p.ep_pos.astype(str) + '/' +
                  p.ep_n.astype(str))
        wp = p.pivot(index='estado', columns='factor',
              values='x').reindex(['sube','baja'])[FACT]
        print(f'\n--- {k} --- media por episodio (%/anio)')
        print(w.round(2).to_string())
        print('episodios positivos / total:')
        print(wp.to_string())
        print('DIFERENCIA sube - baja:')
        print((w.loc['sube'] -
               w.loc['baja']).round(2).to_string())
    print()

eje crec: 12 episodios | duracion media 28.8 meses
crec
sube    221
baja    125 

eje infl: 18 episodios | duracion media 19.2 meses
infl
sube    180
baja    166 

EJE CRECIMIENTO

--- equi --- media por episodio (%/anio)
factor  value  quality  momentum  growth  lowvol
estado                                          
sube     1.31    -1.14     -0.01   -0.32   -9.28
baja    -1.22     7.75      5.95    2.03    8.28
episodios positivos / total:
factor value quality momentum growth lowvol
estado                                     
sube     3/6     3/6      3/6    3/6    1/6
baja     2/6     5/6      4/6    4/6    5/6
DIFERENCIA sube - baja:
factor
value        2.54
quality     -8.90
momentum    -5.96
growth      -2.35
lowvol     -17.56

--- cap --- media por episodio (%/anio)
factor  value  quality  momentum  growth  lowvol
estado                                          
sube     0.42     4.55      1.07    3.95  -11.58
baja     0.04    15.99     10.98    5.50   10.75
episodios positivos

In [ ]:
def patas(peso, eje='crec'):
    out = []
    for f in FACT:
        rq = carteras(d, f, peso=peso)
        for q, nom in [(1,'Q1'), (5,'Q5')]:
            s = rq[q].dropna()
            x = pd.DataFrame({'r': s}).join(mm2)
            x = x.dropna(subset=[eje])
            pe = (x.groupby([eje, 'ep_' + eje]).r.mean()
                   .groupby(level=0).mean() * 12)
            out.append({'factor': f, 'pata': nom,
                        'sube': pe.get('sube'),
                        'baja': pe.get('baja'),
                        'dif': pe.get('sube') - pe.get('baja')})
    return pd.DataFrame(out)

for k, peso in [('equi', None), ('cap', 'mc_live')]:
    print(f'\n=== {k} — eje crecimiento, por pata ===')
    t = patas(peso)
    print(t.pivot(index='factor', columns='pata',
                  values='dif').round(2)
           .reindex(FACT).to_string())


=== equi — eje crecimiento, por pata ===
pata         Q1     Q5
factor                
value     21.06  23.60
quality   26.09  17.20
momentum  26.85  20.89
growth    19.99  17.64
lowvol    29.34  11.78

=== cap — eje crecimiento, por pata ===


/tmp/ipykernel_501/536424770.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda z: z.x.sum()/z.w.sum())
/tmp/ipykernel_501/536424770.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda z: z.x.sum()/z.w.sum())
/tmp/ipykernel_501/536424770.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of p

pata         Q1     Q5
factor                
value     22.63  23.00
quality   29.41  17.97
momentum  30.71  20.80
growth    20.40  18.85
lowvol    36.84  14.51


/tmp/ipykernel_501/536424770.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda z: z.x.sum()/z.w.sum())


In [ ]:

COMP = ['z_bp','z_ey','z_ry',
        'z_sales_3y','z_eps_p',
        'z_roe','z_gpoa','z_bab','z_evol']

SPC = {}
for c in COMP:
    rq = carteras(d, c, peso='mc_live')
    SPC[c] = (rq[5] - rq[1]).dropna()

for eje, nom in [('crec','CRECIMIENTO'),
                 ('infl','INFLACION')]:
    filas = []
    for c in COMP:
        x = pd.DataFrame({'sp': SPC[c]}).join(mm2)
        x = x.dropna(subset=[eje])
        r = {}
        for est in ['sube','baja']:
            g = x[x[eje] == est]
            pe = g.groupby('ep_'+eje).sp.mean()*12
            r[est] = pe.mean()
            r['pos_'+est] = f'{int((pe>0).sum())}/{len(pe)}'
        r['dif'] = r['sube'] - r['baja']
        r['comp'] = c
        filas.append(r)
    t = pd.DataFrame(filas).set_index('comp')
    print(f'\n=== EJE {nom} (cap-weighted) ===')
    print(t[['sube','baja','dif',
             'pos_sube','pos_baja']].round(2).to_string())


/tmp/ipykernel_501/536424770.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda z: z.x.sum()/z.w.sum())
/tmp/ipykernel_501/536424770.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda z: z.x.sum()/z.w.sum())
/tmp/ipykernel_501/536424770.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of p


=== EJE CRECIMIENTO (cap-weighted) ===
            sube   baja    dif pos_sube pos_baja
comp                                            
z_bp       -0.12  -2.65   2.53      2/6      3/6
z_ey       -0.22   6.11  -6.32      2/6      2/6
z_ry        1.44   4.53  -3.09      2/6      3/6
z_sales_3y  5.34  -0.90   6.24      4/6      3/6
z_eps_p     0.91   6.17  -5.26      3/6      5/6
z_roe       2.73  15.33 -12.60      4/6      4/6
z_gpoa      4.22  13.59  -9.37      4/6      5/6
z_bab      -9.72  20.69 -30.41      1/5      5/6
z_evol      0.72   3.25  -2.52      4/6      3/6

=== EJE INFLACION (cap-weighted) ===
             sube   baja    dif pos_sube pos_baja
comp                                             
z_bp         1.52   5.77  -4.25      3/9      5/9
z_ey         8.60   8.99  -0.39      7/9      4/9
z_ry         0.24  11.30 -11.06      3/9      6/9
z_sales_3y   5.29  -7.22  12.51      7/9      4/9
z_eps_p      5.44   3.70   1.74      6/9      6/9
z_roe       11.87   8.01   3.85  

/tmp/ipykernel_501/536424770.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda z: z.x.sum()/z.w.sum())


In [ ]:
#Bloque 6 — retornos sectoriales relativos al mercado y comparación contra Table 6:

s = d.dropna(subset=['fwd1','mc_live','trbc']).copy()
s = s[s.mc_live > 0]
s['x'] = s.fwd1 * s.mc_live

mk = s.groupby('mes').apply(
    lambda g: g.x.sum() / g.mc_live.sum())
sec_r = (s.groupby(['mes','trbc'])
          .apply(lambda g: g.x.sum() / g.mc_live.sum())
          .unstack())
rel = sec_r.sub(mk, axis=0)

mmf = d[['mes','fase','ep']].drop_duplicates()
mmf = mmf.set_index('mes')
x = rel.join(mmf).dropna(subset=['fase'])

tab = (x.groupby(['fase','ep']).mean(numeric_only=True)
        .groupby('fase').mean() * 12).reindex(ORDEN)

print('NUESTRO — retorno relativo al mercado, %/anio')
print(tab.round(1).T.to_string())

/tmp/ipykernel_501/2341823148.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mk = s.groupby('mes').apply(


NUESTRO — retorno relativo al mercado, %/anio
fase                    Reflation  Recovery  Overheat  Stagflation
Basic Materials              -0.3      -6.8       0.6          1.8
Consumer Cyclicals           15.0       6.6      -5.1         -6.8
Consumer Non-Cyclicals        5.6      -2.6      -5.2          2.5
Energy                       -8.1     -20.2       7.9         13.6
Financials                    3.6      -4.9       0.2        -15.9
Healthcare                    8.2     -10.2      -1.5          6.9
Industrials                  -3.5       0.4       0.5          3.0
Real Estate                 -13.5     -12.5       7.7         -6.6
Technology                   -3.4       9.0       2.1          9.5
Utilities                    -1.7     -21.3      -0.8         15.0


/tmp/ipykernel_501/2341823148.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.x.sum() / g.mc_live.sum())


In [ ]:
SEC9 = ['Basic Materials','Consumer Cyclicals',
        'Consumer Non-Cyclicals','Energy',
        'Financials','Healthcare','Industrials',
        'Technology','Utilities']
ML6 = pd.DataFrame(index=SEC9)
ML6['Reflation'] = [0.5, 8.9, 13.3, -12.8,
                    11.0, 5.6, -4.5, -4.6, -4.7]
ML6['Recovery'] = [-2.4, 3.8, -3.1, -4.4,
                   1.4, -4.5, -0.4, 3.3, -3.1]
ML6['Overheat'] = [-3.6, -5.8, 1.1, 4.2,
                   -1.8, 2.9, 4.3, 4.7, -3.2]
ML6['Stagflation'] = [2.1, -8.9, 2.5, 14.7,
                      1.6, 11.6, 2.1, -12.5, 6.4]

print('MERRILL — Table 6')
print(ML6.round(1).to_string())

print('\nCORRELACION DE RANGOS por fase:')
for f in ORDEN:
    a = tab.loc[f].reindex(SEC9)
    b = ML6[f]
    sp = a.corr(b, method='spearman')
    pe = a.corr(b)
    print(f'  {f:12s} spearman {sp:+.2f} '
          f'| pearson {pe:+.2f}')

print('\nSIGNOS QUE COINCIDEN, por sector:')
for sec in SEC9:
    ok = sum(np.sign(tab.loc[f, sec]) ==
             np.sign(ML6.loc[sec, f]) for f in ORDEN)
    print(f'  {sec:24s} {ok}/4')

print('\nPAR ConsDisc - Energia (paper: + con')
print('inflacion bajando, - con inflacion subiendo):')
for f in ORDEN:
    v = (tab.loc[f,'Consumer Cyclicals'] -
         tab.loc[f,'Energy'])
    print(f'  {f:12s} {v:+6.1f}')

MERRILL — Table 6
                        Reflation  Recovery  Overheat  Stagflation
Basic Materials               0.5      -2.4      -3.6          2.1
Consumer Cyclicals            8.9       3.8      -5.8         -8.9
Consumer Non-Cyclicals       13.3      -3.1       1.1          2.5
Energy                      -12.8      -4.4       4.2         14.7
Financials                   11.0       1.4      -1.8          1.6
Healthcare                    5.6      -4.5       2.9         11.6
Industrials                  -4.5      -0.4       4.3          2.1
Technology                   -4.6       3.3       4.7        -12.5
Utilities                    -4.7      -3.1      -3.2          6.4

CORRELACION DE RANGOS por fase:
  Reflation    spearman +0.80 | pearson +0.83
  Recovery     spearman +0.79 | pearson +0.82
  Overheat     spearman +0.48 | pearson +0.52
  Stagflation  spearman +0.54 | pearson +0.36

SIGNOS QUE COINCIDEN, por sector:
  Basic Materials          2/4
  Consumer Cyclicals       4/

In [ ]:
SECS = list(rel.columns)
x2 = rel.join(mm2).dropna(subset=['crec'])

for eje, nom in [('crec','CRECIMIENTO'),
                 ('infl','INFLACION')]:
    res = {}
    for est in ['sube','baja']:
        g = x2[x2[eje] == est]
        pe = g.groupby('ep_' + eje)[SECS].mean()
        res[est] = pe.mean() * 12
        res['n_' + est] = pd.Series(
            len(pe), index=SECS)
    t = pd.DataFrame(res)
    t['dif'] = t.sube - t.baja
    print(f'\n=== EJE {nom} (%/anio) ===')
    print(t[['sube','baja','dif']].round(1)
           .sort_values('dif', ascending=False)
           .to_string())
    print(f"episodios: sube {t.n_sube.iloc[0]} | "
          f"baja {t.n_baja.iloc[0]}")


=== EJE CRECIMIENTO (%/anio) ===
                        sube  baja   dif
Financials               1.9  -7.5   9.5
Real Estate             -0.7  -5.9   5.2
Technology               6.1   2.3   3.8
Industrials             -1.3  -0.9  -0.3
Energy                   0.4   1.4  -0.9
Basic Materials         -3.0   0.9  -3.9
Consumer Cyclicals      -2.0   4.0  -6.0
Healthcare              -4.1   4.2  -8.2
Consumer Non-Cyclicals  -5.8   5.6 -11.4
Utilities               -9.3   9.6 -18.9
episodios: sube 6 | baja 6

=== EJE INFLACION (%/anio) ===
                        sube  baja   dif
Energy                   9.5  -7.9  17.4
Real Estate              0.9  -7.2   8.1
Utilities                0.9  -4.2   5.1
Industrials              1.2  -3.1   4.3
Technology               6.0   2.6   3.4
Basic Materials          1.7   2.2  -0.5
Healthcare              -0.2   2.7  -2.9
Consumer Non-Cyclicals  -2.6   4.4  -7.0
Financials              -5.9   1.7  -7.6
Consumer Cyclicals      -6.2   8.7 -14.9
episo

In [ ]:
MIN_S = 20
filas = []
for sec in SECS:
    g = d[d.trbc == sec]
    for f in FACT:
        z = g.dropna(subset=[f, 'fwd1'])
        ic = (z.groupby('mes')
               .apply(lambda w: w[f].corr(
                   w.fwd1, method='spearman')
                   if len(w) >= MIN_S else np.nan)
               .dropna())
        if len(ic) < 24:
            continue
        icir = ic.mean() / ic.std()
        filas.append({'sector': sec, 'factor': f,
            'IC': ic.mean(), 'ICIR': icir,
            't': icir * np.sqrt(len(ic)),
            'n': len(ic)})
C = pd.DataFrame(filas)

print('IC medio por sector y factor:')
print(C.pivot(index='sector', columns='factor',
      values='IC').reindex(columns=FACT)
      .round(4).to_string())

print('\nt-stat (umbral honesto |t|>3.3):')
print(C.pivot(index='sector', columns='factor',
      values='t').reindex(columns=FACT)
      .round(2).to_string())

print('\nmeses utilizables por sector:')
print(C.groupby('sector').n.max().to_string())

print('\nceldas con |t|>3.3:')
fuerte = C[C.t.abs() > 3.3].sort_values(
    't', ascending=False)
print(fuerte[['sector','factor','IC','t','n']]
      .round(3).to_string(index=False)
      if len(fuerte) else '  ninguna')

/tmp/ipykernel_501/3704994971.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda w: w[f].corr(
/tmp/ipykernel_501/3704994971.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda w: w[f].corr(
/tmp/ipykernel_501/3704994971.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping c

IC medio por sector y factor:
factor                   value  quality  momentum  growth  lowvol
sector                                                           
Basic Materials        -0.0031   0.0194    0.0231  0.0008  0.0043
Consumer Cyclicals      0.0003   0.0167    0.0314  0.0188  0.0148
Consumer Non-Cyclicals  0.0141   0.0072   -0.0013  0.0065 -0.0042
Energy                  0.0301   0.0109    0.0071 -0.0065  0.0180
Financials              0.0309   0.0020    0.0062  0.0307 -0.0051
Healthcare              0.0211   0.0037    0.0060 -0.0055  0.0094
Industrials             0.0159   0.0039    0.0064 -0.0038  0.0156
Real Estate            -0.0069   0.0101    0.0195  0.0145 -0.0208
Technology             -0.0136   0.0243    0.0084  0.0105  0.0034
Utilities               0.0125  -0.0234    0.0189 -0.0257 -0.0128

t-stat (umbral honesto |t|>3.3):
factor                  value  quality  momentum  growth  lowvol
sector                                                          
Basic Material

/tmp/ipykernel_501/3704994971.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda w: w[f].corr(
